In [1]:
import os
import sys

# Tell Spark to use the lightning-fast native Linux environment
os.environ['PYSPARK_PYTHON'] = '/home/harsh/wsl_venv/bin/python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/harsh/wsl_venv/bin/python3'

In [2]:
from pyspark.sql import SparkSession
from delta import *

In [3]:
builder = SparkSession.builder \
    .appName("Local_Lakehouse_70GB_GPU") \
    .master("local[3]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.caseSensitive", "true")

In [4]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 00:53:15 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/12 00:53:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86914fc0-5d51-4da4-a4f1-e9a3e029f3bd;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

### Reading data

In [5]:
post_df = spark.read \
    .format("parquet") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/readyForSentimentAnalysis/")

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [7]:
post_df.show(n=10,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
from typing import Iterator
import pandas as pd
from transformers import pipeline
import torch

In [8]:
# 1. Create a tiny, lightning-fast PyTorch Dataset wrapper
class ListDataset(torch.utils.data.Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, i):
        return self.data_list[i]

@pandas_udf(FloatType())
def roberta_sentiment_udf(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    
    analyzer = pipeline(
        "sentiment-analysis", 
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        device=0, 
        batch_size=64, 
        truncation=True,
        max_length=512
    )
    batch_idx=0
    
    for text_batch in iterator:
        # 2. Wrap the Pandas Series inside our PyTorch Dataset
        dataset = ListDataset(text_batch.tolist())
        print(f"done batch_idx:{batch_idx} | len(text_batch):{len(text_batch)}")
        
        # 3. Feed the Dataset to the analyzer (This silences the warning!)
        # The analyzer returns a generator when given a Dataset, so we loop through it
        final_scores = []
        for pred in analyzer(dataset):
            if pred['label'] == 'positive':
                final_scores.append(pred['score'])
            elif pred['label'] == 'negative':
                final_scores.append(-pred['score'])
            else:
                final_scores.append(0.0)
                
        yield pd.Series(final_scores)

In [9]:
sentimentTable = post_df.withColumn("sentiment_score", roberta_sentiment_udf(col("commit.record.text")))

In [10]:
# sentimentTable.select(col("commit.record.text").alias("text"),col("sentiment_score")).show(n=100,truncate=False)

In [11]:
sentimentTable.printSchema()

root
 |-- _corrupt_record: string (nullable = true)
 |-- account: struct (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- did: string (nullable = true)
 |    |-- seq: long (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- commit: struct (nullable = true)
 |    |-- cid: string (nullable = true)
 |    |-- collection: string (nullable = true)
 |    |-- operation: string (nullable = true)
 |    |-- record: struct (nullable = true)
 |    |    |-- $type: string (nullable = true)
 |    |    |-- $via: string (nullable = true)
 |    |    |-- 0: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- \: string (nullable = true)
 |    |    |-- \$type: string (nullable = true)
 |    |    |-- actor: string (nullable = true)
 |    |    |-- app.photosky.v: long (nullable = true)
 |    |    |-- bridgyOriginalText: string (nullable = true)
 |    |    |-- bridgyOriginalUrl: string (nu

In [ ]:
sentimentTable.write \
    .format("parquet") \
    .mode("append") \
    .save("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/sentimentTable")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15041.93it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you e

KeyboardInterrupt: 

26/04/11 12:25:32 ERROR Utils: Aborting task
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_163410/3099574228.py", line 31, in roberta_sentiment_udf
  File "/home/harsh/wsl_venv/lib/python3.12/site-packages/transformers/pipelines/pt_utils.py", line 126, in __next__
    item = next(self.iterator)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/harsh/wsl_venv/lib/python3.12/site-packages/transformers/pipelines/pt_utils.py", line 127, in __next__
    processed = self.infer(item, **self.params)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/harsh/wsl_venv/lib/python3.12/site-packages/transformers/pipelines/base.py", line 1164, in forward
    model_outputs = self._ensure_tensor_on_device(model_outputs, device=torch.device("cpu"))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/harsh/wsl_venv/lib/python3.12/site-packages/transformers/pipelines/base.py", line 1064, in

In [13]:
post_df.count()

26/04/11 12:25:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


17448912

In [ ]:
# sentimentTable=sentimentTable.withColumn("splitted_text",array_distinct(split(lower(col("commit.record.text")),r"\s+")))

In [ ]:
# sentimentTable=sentimentTable.withColumn("exploded_text",explode(col("splitted_text")))

In [ ]:
# sentimentTable=sentimentTable.groupBy("exploded_text").agg(avg(col("sentiment_score")))

In [ ]:
# sentimentTable.show(n=100,truncate=False)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12201.04it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------